# Beta-Neutral Dynamic Strategy

A **beta-neutral** strategy that dynamically adjusts portfolio weights to maintain
near-zero market beta, using `Weigh.BasedOnBeta`.

The strategy scales an initial allocation so that the portfolio's beta to a reference
asset (AAPL) is approximately zero. High-beta assets get scaled down; low/negative-beta
assets get scaled up.

This notebook demonstrates:
- `Weigh.BasedOnBeta` for beta-targeted allocation
- Dynamic weight adjustment based on rolling beta estimates
- Weight evolution visualisation
- Comparison against static baselines

**Offline mode**: Uses local CSV files — no network required.

In [ ]:
from pathlib import Path

import pandas as pd

import tiportfolio as ti

_DATA_DIR = Path("../../tests/data")

CSV_DATA: dict[str, str] = {
    "QQQ": str(_DATA_DIR / "qqq_2018_2024_yf.csv"),
    "BIL": str(_DATA_DIR / "bil_2018_2024_yf.csv"),
    "GLD": str(_DATA_DIR / "gld_2018_2024_yf.csv"),
    "AAPL": str(_DATA_DIR / "aapl_2018_2024_yf.csv"),
}

TICKERS = ["QQQ", "BIL", "GLD"]
INITIAL_RATIO = {"QQQ": 0.7, "BIL": 0.2, "GLD": 0.1}
START = "2019-01-01"
END = "2024-12-31"

## 1. Load Data

AAPL is used as the beta reference (base_data) — not traded in the portfolio.

In [ ]:
data = ti.fetch_data(TICKERS, start=START, end=END, csv=CSV_DATA)
aapl_data = ti.fetch_data(["AAPL"], start=START, end=END, csv=CSV_DATA)

for ticker, df in data.items():
    print(f"{ticker}: {df.shape[0]} rows, {df.index[0].date()} \u2192 {df.index[-1].date()}")
print(f"AAPL (reference): {aapl_data['AAPL'].shape[0]} rows")

## 2. Beta-Neutral Strategy

`Weigh.BasedOnBeta` iteratively scales the initial weights so that the portfolio's
beta to the reference asset approaches the target (0 = market-neutral).

- Uses rolling OLS regression over the lookback window to estimate betas
- Weights are NOT normalised — sum(w) < 1 implies a cash residual
- Monthly rebalancing ensures beta stays near zero as market conditions change

In [ ]:
beta_neutral = ti.Portfolio(
    "beta_neutral",
    [
        ti.Signal.Monthly(),
        ti.Select.All(),
        ti.Weigh.BasedOnBeta(
            initial_ratio=INITIAL_RATIO,
            target_beta=0,
            lookback=pd.DateOffset(months=1),
            base_data=aapl_data["AAPL"],
        ),
        ti.Action.Rebalance(),
    ],
    TICKERS,
)

result = ti.run(ti.Backtest(beta_neutral, data))

## 3. Results

In [ ]:
result.summary()

In [ ]:
result[0].plot_interactive()

In [ ]:
result.full_summary()

## 4. Weight Evolution

Watch how weights shift over time as the strategy adjusts to maintain zero beta:

In [ ]:
result[0].plot_security_weights()

In [ ]:
result[0].plot_histogram()

In [ ]:
print(f"Total trades: {len(result[0].trades)}")
result[0].trades.sample(5)

## 5. Baseline Comparisons

Compare the beta-neutral strategy against:
1. **Equal Weight** — equal allocation across QQQ/BIL/GLD, monthly rebalance
2. **QQQ Only** — 100% QQQ buy-and-hold

In [ ]:
equal_weight = ti.Portfolio(
    "equal_weight",
    [ti.Signal.Monthly(), ti.Select.All(), ti.Weigh.Equally(), ti.Action.Rebalance()],
    TICKERS,
)

qqq_only = ti.Portfolio(
    "qqq_only",
    [ti.Signal.Once(), ti.Select.All(), ti.Weigh.Equally(), ti.Action.Rebalance()],
    ["QQQ"],
)

comparison = ti.run(
    ti.Backtest(beta_neutral, data),
    ti.Backtest(equal_weight, data),
    ti.Backtest(qqq_only, data),
)

In [ ]:
comparison.summary()

In [ ]:
comparison.plot_interactive()

In [ ]:
comparison.plot_histogram()